In [ ]:
from functools import partial
from pathlib import Path

import s3fs

import numpy as np
import xarray as xr
import pint_xarray
import pint
import matplotlib as mpl
import matplotlib.pylab as plt
import pandas as pd
import geopandas as gpd

from pism_terra.processing import preprocess_netcdf, normalize_timeseries, integrate_rate
from pism_terra.plotting import rc_params
from pism_terra.postprocess_scalar import basin_masks
from pism_terra.workflow import dataset_crs, drop_grid_mapping

In [ ]:
ssps = { "historical": "k",
         "OCX": "k",
         "ctrl": "0.5",
         "ssp119": "#00ADCF",
         "ssp126": "#173C66",
         "ssp245": "#F79420",
         "ssp370": "#E71D25",
         "ssp585": "#951B1E"}
gcms = {"MRI-ESM2-0": "dashed", "CESM2-WACCM": "solid"}

In [ ]:
reference_year = "1990"

ORDER = ["GIS_GIS", "GIS_NO", "GIS_NE", "GIS_CE", "GIS_SE", "GIS_SW", "GIS_CW", "GIS_NW"]

In [ ]:
ROOT = Path("/Users/andy/base/pism-terra/2026_10_ismip7_core_ctrl/output/GrIS/UAF/PISM/CORE")

# <var>_GrIS_UAF_PISM_m001_<gcm>_f001_<exp>_<Cxxx>_<start>-<end>.nc
preprocess_ismip7 = partial(
    preprocess_netcdf,
    gcm_dim="gcm_id",
    gcm_regexp=r"_m\d+_(.+?)_f\d+_",
    exp_dim="ssp_id",
    exp_regexp=r"_f\d+_(.+?)_C\d+_",
    rgi_dim=None,
    uq_dim=None,
    process_config=False,      # submission files carry no pism_config
)


def nonempty(paths):
    """Drop files whose time axis is still empty (run in flight / rsync running)."""
    keep = []
    for p in paths:
        with xr.open_dataset(p, decode_times=False, decode_timedelta=False) as ds:
            if ds.sizes.get("time", 1) > 0:
                keep.append(p)
    return keep


def open_ismip7(paths, **kwargs) -> xr.Dataset:
    return xr.open_mfdataset(
        nonempty(paths),
        preprocess=preprocess_ismip7,
        combine="by_coords",
        chunks={},                     # dask-backed, one chunk per file
        decode_timedelta=True,
        data_vars="minimal",
        coords="minimal",
        compat="override",
        combine_attrs="drop_conflicts",
        join="outer",                  # (gcm, ssp) combos never run -> N
        parallel=True,
        **kwargs,
    )


def splice_historical(ds: xr.Dataset, historical: str = "historical") -> xr.Dataset:
    """Prepend the historical member to every ssp of the same GCM (still lazy)."""
    hist = ds.sel(ssp_id=historical, drop=True)
    return ds.drop_sel(ssp_id=historical).combine_first(hist)



VARS = ["acabf", "ligroundf", "libmassbfgr"]
paths = sorted(p for v in VARS for p in ROOT.glob(f"C*/{v}_*.nc"))
ds = open_ismip7(paths)

outline_file = "/Users/andy/base/pism-terra/ismip7_input/mouginot_basins_w_shelves.gpkg"

In [ ]:
def ismip7_crs(ds, crs=None):
    """dataset_crs wants crs_wkt; submission files carry mapping.proj_params instead."""
    try:
        return dataset_crs(ds, crs)
    except ValueError:
        fallback = ds.attrs.get("crs") or ds.get("mapping", xr.DataArray()).attrs.get("proj_params")
        if not fallback:
            raise
        return fallback


def regional_sums(ds, outlinefile, column="glacier_id", dim_name="region",
                  crs=None, all_touched=False, client=None):
    """Sum every (x, y) field over each outline; stays lazy."""
    dst_crs = ismip7_crs(ds, crs)
    outline = gpd.GeoDataFrame(
        gpd.read_file(outlinefile).to_crs(dst_crs),
        crs=dst_crs,
    )

    spatial_vars = [v for v in ds.data_vars if {"x", "y"} <= set(ds[v].dims)]
    grid = ds[spatial_vars].rio.write_crs(dst_crs).rio.set_spatial_dims(x_dim="x", y_dim="y")
    cell_area = abs(float(ds.x[1] - ds.x[0]) * float(ds.y[1] - ds.y[0]))

    masks = basin_masks(grid, outline, column=column, all_touched=all_touched, client=client)
    out = xr.concat(
        [(grid.where(mask).sum(dim=["y", "x"]) * cell_area).expand_dims({dim_name: [name]})
         for name, mask in masks],
        dim=dim_name,
    )

    # kg m-2 s-1 summed over cells x m^2 = kg s-1; pint does the bookkeeping because
    # the files spell it several ways ("kg m^-2 s^-1", "kg m^-2 second^-1")
    ureg = pint.application_registry
    for var in out.data_vars:
        if units := ds[var].attrs.get("units"):
            out[var].attrs["units"] = f"{(ureg(units) * ureg('m^2')).units:~}"

    out["area"] = xr.DataArray(
        [geom.area for geom in outline.geometry], dims=(dim_name,),
        coords={dim_name: out[dim_name]}, attrs={"units": "m^2", "long_name": "outline area"},
    )    
    out = xr.concat([out, out.sum("region").expand_dims(region=["GIS_GIS"])], "region")
    out = drop_grid_mapping(out)
    return out


def to_gt_per_year(ds, target="Gt/year"):
    """Convert every variable whose units allow it; leave the rest alone."""
    ureg = pint.application_registry
    conv = [v for v in ds.data_vars
            if ds[v].attrs.get("units") and ureg(ds[v].attrs["units"]).is_compatible_with(target)]
    out = ds[conv].pint.quantify().pint.to({v: target for v in conv}).pint.dequantify()
    return out.assign({v: ds[v] for v in ds.data_vars if v not in conv})
    

regions = to_gt_per_year(regional_sums(ds, outline_file).compute())
regions["mass_balance"] = regions["acabf"] +  regions["libmassbfgr"] + regions["ligroundf"]
regions["cumulative_mass_balance"] = normalize_timeseries(integrate_rate(regions["mass_balance"]),  reference_date=reference_year)
regions = regions.reindex(region=[r for r in ORDER if r in set(regions.region.values)])


In [ ]:
PAT = "s3://pism-cloud-data/ismip7_production/2026_09_core/*/output/observations/mankoff_greenland_mass_balance.nc"

fs = s3fs.S3FileSystem(anon=True)
urls = [f"s3://{k}" for k in fs.glob(PAT)]          # note the extra */ level
mankoff_url = urls[0]

mankoff = xr.open_dataset(mankoff_url, 
                          decode_times=xr.coders.CFDatetimeCoder(),
                          decode_timedelta=xr.coders.CFTimedeltaCoder(),
                          engine="h5netcdf", storage_options={"anon": True}, chunks={"time": -1}).pint.quantify()
mankoff = mankoff.pint.to({"cumulative_mass_balance": "Gt", "cumulative_mass_balance_uncertainty": "Gt"}).pint.dequantify()
mankoff = mankoff.assign_coords(region=np.char.add("GIS_", mankoff.region.values))
mankoff = mankoff.sel({"time": slice("1980", "2025")}).resample(time='YS').mean('time')
mankoff = normalize_timeseries(mankoff, variables=["cumulative_mass_balance", "cumulative_mass_balance_uncertainty"], reference_date=reference_year)

In [ ]:
sigma = 1
rc_params_small = rc_params.copy()
rc_params["font.size"] = 4
with mpl.rc_context(rc=rc_params):

    fig, axs = plt.subplots(2, 4, figsize=(6.4, 3.6), sharex=True, sharey=False, layout="constrained")
    for k, region in enumerate(regions.sel(region=ORDER).region.values):
        region_ds = regions.sel(region=region)
        ax = axs.flatten()[k]
        mankoff_region = mankoff.sel(region=region)

        ax.fill_between(mankoff_region.time, 
                        mankoff_region.cumulative_mass_balance - sigma * mankoff_region.cumulative_mass_balance_uncertainty, 
                        mankoff_region.cumulative_mass_balance + sigma * mankoff_region.cumulative_mass_balance_uncertainty,
                        lw=0, color="0.75", alpha=0.5)
        # mankoff_region.cumulative_mass_balance.plot(ax=ax, lw=0.75, color="k", ls="solid")
        for (_gcm, _ssp), _data in region_ds.groupby(["gcm_id", "ssp_id"]):
            _data.cumulative_mass_balance.plot(ax=ax, ls=gcms[_gcm], color=ssps[_ssp], lw=0.5)
        ax.set_xlim(np.datetime64("1980"), np.datetime64("2100"))
        ax.set_xlabel(None)
        ax.set_ylabel(None)
        ax.set_title(region)
        
    fig.supxlabel("Year")
    fig.supylabel(f"Cumulative mass change since {reference_year} (Gt)")
    fig.savefig(f"ismip7_greenland_pism.png", dpi=300)
    plt.show()
    plt.close(fig)

In [ ]:
sigma = 1
for _region in regions.region:
    region = _region.values
    region_ds = regions.sel(region=region)
    mankoff_region = mankoff.sel(region=region)

    with mpl.rc_context(rc=rc_params):

        fig, ax = plt.subplots(1, 1, figsize=(6.2, 4.2), sharex=True)
        ax.fill_between(mankoff_region.time, 
                        mankoff_region.cumulative_mass_balance - sigma * mankoff_region.cumulative_mass_balance_uncertainty, 
                        mankoff_region.cumulative_mass_balance + sigma * mankoff_region.cumulative_mass_balance_uncertainty,
                        lw=0, color="0.75", alpha=0.5)
        mankoff_region.cumulative_mass_balance.plot(ax=ax, lw=1., color="k", ls="solid")
        region_ds.cumulative_mass_balance.plot()
        ax.set_xlim(np.datetime64("1980"), np.datetime64("2300"))
        
        fig, axs = plt.subplots(3, 1, figsize=(6.2, 4.2), sharex=True)
        axs[0].fill_between(mankoff_region.time, 
                        mankoff_region.mass_balance - sigma * mankoff_region.mass_balance_uncertainty, 
                        mankoff_region.mass_balance + sigma * mankoff_region.mass_balance_uncertainty,
                        lw=0, color="0.75", alpha=0.5)
        axs[1].fill_between(mankoff_region.time, 
                        mankoff_region.surface_mass_balance - sigma * mankoff_region.surface_mass_balance_uncertainty, 
                        mankoff_region.surface_mass_balance + sigma * mankoff_region.surface_mass_balance_uncertainty,
                        lw=0, color="0.75", alpha=0.5)    
        axs[2].fill_between(mankoff_region.time, 
                        mankoff_region.grounding_line_flux - sigma * mankoff_region.grounding_line_flux_uncertainty, 
                        mankoff_region.grounding_line_flux + sigma * mankoff_region.grounding_line_flux_uncertainty,
                        lw=0, color="0.75", alpha=0.5)    
        mankoff_region.mass_balance.plot(ax=axs[0], lw=1., color="k", ls="solid")
        mankoff_region.surface_mass_balance.plot(ax=axs[1], lw=1, color="k", ls="dashed")
        mankoff_region.grounding_line_flux.plot(ax=axs[2], lw=1, color="k", ls="dotted")
            
        axs[0].set_prop_cycle(None)
        axs[0].set_xlabel(None)
        axs[0].set_title(None)

        axs[1].set_prop_cycle(None)
        axs[1].set_xlabel(None)
        axs[1].set_title(None)

        axs[2].set_prop_cycle(None)
        axs[-1].set_title(None)
        axs[-1].set_xlim(np.datetime64("1980"), np.datetime64("2025"))
        fig.tight_layout()
        fig.savefig(f"ismip7_greenland_fluxes_{region}.png", dpi=300)
        plt.show()
        plt.close(fig)

In [ ]:
!open ismip7_greenland_pism.png

In [ ]:
for (_gcm, _ssp), _d in region_ds.groupby(["gcm_id", "ssp_id"]):
    print(_d)

In [ ]:
_d.cumulative_mass_balance.time

In [ ]:
mankoff

In [ ]:
_d

In [ ]:
regions.region

In [ ]:
regions.time